# TabICLv2 (Inria) — Classification

Demonstrates **TabICL**, the zero-shot tabular foundation model, on the shared
retail/CPG classification datasets. TabICLv2 is **self-hosted**: model weights are
downloaded from Hugging Face and inference runs locally in a single forward pass.

> **License note:** TabICLv2 is released under a [commercially permissive license](https://huggingface.co/datasets/choosealicense/licenses/blob/main/markdown/bsd-3-clause.md) 

**Compute:** GPU cluster recommended (weights run on GPU). If GPU serverless used, a single 1xA10 is enough. CPU works but is slow.

**Prerequisite:** run [`shared/notebooks/00_data_preparation.ipynb`](../../../shared/notebooks/00_data_preparation.ipynb) first.


In [0]:
%pip install tabicl==2.1.1 --quiet

In [0]:
dbutils.library.restartPython()

## Configuration

Catalog/schema must match the shared data-prep notebook. `common/` is added to the
path so we can import the shared evaluation helpers.


In [0]:
import os, sys

# Make the repo-level common/ package importable.
# Adjust REPO_ROOT if your repo is checked out at a different workspace path.
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
COMMON_PATH = os.path.join(REPO_ROOT, "common")
if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from config import CATALOG

# Configure catalog and schema (must match shared/notebooks/00_data_preparation, imported from common/config.py)
SCHEMA = "default"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# MLflow experiment (shared naming convention across vendors)
current_user = spark.sql("SELECT current_user()").collect()[0][0]
MLFLOW_EXPERIMENT_NAME = f"/Users/{current_user}/tabular-fm-databricks"

print(f"Catalog/schema: {CATALOG}.{SCHEMA}")
print(f"common/ path:   {COMMON_PATH}  (exists={os.path.isdir(COMMON_PATH)})")

## Import libraries

TabICLv2 downloads the weights from HuggingFace directly when loading train data into context using an sklearn-style estimator (i.e. .fit)


In [0]:
import numpy as np
import pandas as pd
import mlflow

from sklearn.preprocessing import OrdinalEncoder

import torch
from tabicl import TabICLClassifier

from evaluation import (
    split_xy, classification_metrics, train_baselines_classification,
    log_result, RESULTS_TABLE,
)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)


In [0]:
def evaluate_classification(table_name, target, problem_type, task_name,
                            test_size=0.2, stratify=True):
    df = spark.table(table_name).toPandas()
    X_train, X_test, y_train, y_test = split_xy(
        df, target=target, test_size=test_size, stratify=stratify
    )
    n_train, n_test, n_features = len(X_train), len(X_test), X_train.shape[1]

    # Encode categorical columns (TabICL requires numeric inputs)
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    if cat_cols:
        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        X_train = X_train.copy()
        X_test = X_test.copy()
        X_train[cat_cols] = enc.fit_transform(X_train[cat_cols])
        X_test[cat_cols] = enc.transform(X_test[cat_cols])

    # --- TabICL (zero-shot forward pass) ---
    with mlflow.start_run(run_name=f"{task_name}_tabicl"):
        mlflow.log_params({
            "vendor": "tabicl", "model_type": "TabICLClassifier",
            "task": task_name, "problem_type": problem_type,
            "n_features": n_features, "train_samples": n_train, "test_samples": n_test,
        })
        device = "cuda" if torch.cuda.is_available() else "cpu"
        clf = TabICLClassifier(device=device)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_pred_proba = clf.predict_proba(X_test)
        metrics = classification_metrics(y_test, y_pred, y_pred_proba)
        mlflow.log_metrics({k: v for k, v in metrics.items() if v is not None})
        log_result(spark, vendor="tabicl", task=task_name, problem_type=problem_type,
                   model_name="TabICLClassifier", metrics=metrics,
                   n_train=n_train, n_test=n_test, n_features=n_features)
    print(f"[{task_name}] TabICL: acc={metrics['accuracy']:.4f} "
          f"f1={metrics['f1']:.4f} roc_auc={metrics['roc_auc']}")

    # --- Shared baselines (identical split) ---
    for name, m in train_baselines_classification(X_train, y_train, X_test, y_test).items():
        log_result(spark, vendor="baseline", task=task_name, problem_type=problem_type,
                   model_name=name, metrics=m,
                   n_train=n_train, n_test=n_test, n_features=n_features)
        print(f"[{task_name}] {name}: acc={m['accuracy']:.4f} f1={m['f1']:.4f}")
    return metrics

## Binary classification — Supplier Delay Risk


In [0]:
_ = evaluate_classification(
    table_name="supplier_delay_risk_train",
    target="is_delayed",
    problem_type="binary_classification",
    task_name="supplier_delay_risk",
    test_size=0.2, stratify=True,
)

## Multi-class classification — Material Shortage

TabFM supports classification with up to 10 classes; these tasks have 3.


In [0]:
_ = evaluate_classification(
    table_name="material_shortage_train",
    target="shortage_risk",
    problem_type="multiclass_classification",
    task_name="material_shortage",
    test_size=0.3, stratify=True,
)

## Multi-class classification — OTIF Risk


In [0]:
_ = evaluate_classification(
    table_name="otif_risk_train",
    target="otif_risk",
    problem_type="multiclass_classification",
    task_name="otif_risk",
    test_size=0.3, stratify=True,
)

## Results

All rows for this run are in the shared results table. The comparison notebook
aggregates across vendors.


In [0]:
display(
    spark.table(RESULTS_TABLE)
         .where("problem_type LIKE '%classification%'")
         .orderBy("task", "vendor", "model_name")
)